In [3]:
!python3 -m pip install pandas numpy scikit-learn matplotlib yellowbrick joblib

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 47 kB/s eta 0:00:011
     |████████████████████████████████| 5.3 MB 176 kB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 853 kB/s eta 0:00:01
     |████████████████████████████████| 7.8 MB 1.7 MB/s eta 0:00:01
     |████████████████████████████████| 282 kB 814 kB/s eta 0:00:01
     |████████████████████████████████| 309 kB 875 kB/s eta 0:00:01
     |████████████████████████████████| 348 kB 702 kB/s eta 0:00:01    |█▉                              | 20 kB 523 kB/s eta 0:00:01
     |████████████████████████████████| 506 kB 431 kB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 72 kB/s eta 0:00:011    |██████████▏                     | 9.6 MB 101 kB/s eta 0:03:26     |██████████████▎                 | 13.5 MB 445 kB/s eta 0:00:38     |████████████████                | 15.1 MB 196 kB/s eta 0:01:18
     |████████████████████████████████| 

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load pre-split datasets from the data/ folder
train_df = pd.read_csv('data/train.csv')
val_df   = pd.read_csv('data/val.csv')
test_df  = pd.read_csv('data/test.csv')

# Standardize column names to lowercase
for d in [train_df, val_df, test_df]:
    d.columns = d.columns.str.lower().str.strip()

print(f"Dataset shapes -> Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

Dataset shapes -> Train: (8393, 10) | Val: (1799, 10) | Test: (1799, 10)


In [2]:
X_train, y_train = train_df.drop('left', axis=1), train_df['left']
X_val, y_val     = val_df.drop('left', axis=1), val_df['left']
X_test, y_test   = test_df.drop('left', axis=1), test_df['left']

ordinal_cols = ['salary']
nominal_cols = ['departments']
numeric_cols = [c for c in X_train.columns if c not in ordinal_cols + nominal_cols]


preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=[['low', 'medium', 'high']]), ordinal_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_processed)
X_val_scaled   = final_scaler.transform(X_val_processed)
X_test_scaled  = final_scaler.transform(X_test_processed)

print("-> Preprocessing completed: Applied Ordinal & One-Hot Encoding. No Data Leakage.")

-> Preprocessing completed: Applied Ordinal & One-Hot Encoding. No Data Leakage.


In [3]:
tune_seeds = [10, 11, 12]

param_grids = {
    "KNN": {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance']
    },
    "Random Forest": {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5]
    },
    "Gradient Boosting": {
        'n_estimators': [100, 150],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    }
}

best_params = {}
print("=== HYPERPARAMETER TUNING ON VALIDATION SET ===")

best_knn_f1 = -1
for params in ParameterGrid(param_grids["KNN"]):
    model = KNeighborsClassifier(**params)
    model.fit(X_train_scaled, y_train)
    val_preds = model.predict(X_val_scaled)
    f1 = f1_score(y_val, val_preds)
    if f1 > best_knn_f1:
        best_knn_f1 = f1
        best_params["KNN"] = params

print(f"[KNN] Best Params: {best_params['KNN']} | Max Val F1: {best_knn_f1:.4f}")

for model_name, ModelClass in [("Random Forest", RandomForestClassifier), ("Gradient Boosting", GradientBoostingClassifier)]:
    best_mean_f1 = -1
    for params in ParameterGrid(param_grids[model_name]):
        seed_f1_scores = []
        for seed in tune_seeds:
            model = ModelClass(**params, random_state=seed)
            model.fit(X_train_scaled, y_train)
            val_preds = model.predict(X_val_scaled)
            seed_f1_scores.append(f1_score(y_val, val_preds))
        
        mean_f1 = np.mean(seed_f1_scores)
        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_params[model_name] = params
            
    print(f"[{model_name}] Best Params: {best_params[model_name]} | Mean Val F1 (3 seeds): {best_mean_f1:.4f}")

=== HYPERPARAMETER TUNING ON VALIDATION SET ===
[KNN] Best Params: {'n_neighbors': 5, 'weights': 'uniform'} | Max Val F1: 0.8744
[Random Forest] Best Params: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 100} | Mean Val F1 (3 seeds): 0.9607
[Gradient Boosting] Best Params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100} | Mean Val F1 (3 seeds): 0.9486


In [6]:
eval_seeds = [42, 43, 44, 45, 46]
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

def compute_metrics(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else y_pred
    return {
        'Accuracy': accuracy_score(y_eval, y_pred),
        'Precision': precision_score(y_eval, y_pred, zero_division=0),
        'Recall': recall_score(y_eval, y_pred, zero_division=0),
        'F1-Score': f1_score(y_eval, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, y_prob)
    }

final_report_rows = []

# 1. Baseline: Dummy Classifier (1 Seed)
dummy_model = DummyClassifier(strategy='stratified', random_state=42)
dummy_model.fit(X_train_scaled, y_train)
dummy_res = compute_metrics(dummy_model, X_test_scaled, y_test)
final_report_rows.append({
    'Model': 'Dummy Classifier (Baseline)',
    **{m: f"{dummy_res[m]:.4f} ± 0.0000" for m in metrics_list}
})

# 2. Tuned KNN (1 Seed)
knn_model = KNeighborsClassifier(**best_params["KNN"])
knn_model.fit(X_train_scaled, y_train)
knn_res = compute_metrics(knn_model, X_test_scaled, y_test)
final_report_rows.append({
    'Model': 'K-Nearest Neighbors (Tuned)',
    **{m: f"{knn_res[m]:.4f} ± 0.0000" for m in metrics_list}
})

# 3. Tuned Random Forest & Gradient Boosting (Evaluated across 5 New Seeds)
best_rf_model = None
for model_name, ModelClass in [("Gradient Boosting", GradientBoostingClassifier), ("Random Forest", RandomForestClassifier)]:
    seed_metrics = {m: [] for m in metrics_list}
    
    for seed in eval_seeds:
        model = ModelClass(**best_params[model_name], random_state=seed)
        model.fit(X_train_scaled, y_train)
        res = compute_metrics(model, X_test_scaled, y_test)
        
        for m in metrics_list:
            seed_metrics[m].append(res[m])
            
        if model_name == "Random Forest" and seed == eval_seeds[0]:
            best_rf_model = model
            
    row = {'Model': f"{model_name} (Tuned, 5 Seeds)"}
    for m in metrics_list:
        mean_val = np.mean(seed_metrics[m])
        std_val  = np.std(seed_metrics[m])
        row[m]   = f"{mean_val:.4f} ± {std_val:.4f}"
    final_report_rows.append(row)

# Format and display the final comparison table cleanly
report_df = pd.DataFrame(final_report_rows)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n" + "=" * 105)
print(" FINAL TEST SET EVALUATION REPORT (MEAN ± STD)")
print("=" * 105)
print(report_df.to_string(index=False))

# Export models and preprocessing objects
joblib.dump(best_rf_model, 'best_employee_churn_model.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(final_scaler, 'scaler.pkl')
print("\n-> Saved model artifacts (*.pkl) successfully!")


 FINAL TEST SET EVALUATION REPORT (MEAN ± STD)
                             Model        Accuracy       Precision          Recall        F1-Score         ROC-AUC
       Dummy Classifier (Baseline) 0.7065 ± 0.0000 0.1217 ± 0.0000 0.1242 ± 0.0000 0.1229 ± 0.0000 0.4731 ± 0.0000
       K-Nearest Neighbors (Tuned) 0.9450 ± 0.0000 0.8024 ± 0.0000 0.8859 ± 0.0000 0.8421 ± 0.0000 0.9529 ± 0.0000
Gradient Boosting (Tuned, 5 Seeds) 0.9839 ± 0.0000 0.9654 ± 0.0000 0.9362 ± 0.0000 0.9506 ± 0.0000 0.9828 ± 0.0000
    Random Forest (Tuned, 5 Seeds) 0.9840 ± 0.0004 0.9787 ± 0.0001 0.9235 ± 0.0025 0.9503 ± 0.0014 0.9779 ± 0.0025

-> Saved model artifacts (*.pkl) successfully!
